In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


df = pd.read_csv("/content/drive/MyDrive/emoji_sample.csv")
lbl = sorted(df["label"].unique())



In [ ]:
llm = "mistralai/Mistral-7B-Instruct-v0.2"

tokenize = AutoTokenizer.from_pretrained(llm)

m = AutoModelForCausalLM.from_pretrained(llm, device_map="auto", torch_dtype="auto")


generator=pipeline("text-generation", model=m, tokenizer=tokenize, max_new_tokens=32, temperature=0, do_sample=False)




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
lbl = sorted(df["label"].unique())


def prompt_eng(txt, lbl):
    lbl2 = ", ".join(lbl)
    prompt = (
        "Choose the correct emoji label for this tweet.\n"
        f"Available labels: {lbl2}\n"
        f"Tweet: {txt}\n"
        "Answer with only one label:"
    )
    return prompt


In [ ]:
def predictor(txt, lbl):
    p = prompt_eng(txt, lbl)
    gen = generator(p)[0]["generated_text"]

    comp = gen[len(p):].strip()

    first = comp.split("\n")[0].strip()

    tkn = first.split()[-1].strip('"').strip("'").strip(".,:")

    if tkn in lbl:
        return tkn

    for l in lbl:
        if l in first:
            return l

    return lbl[0]


In [ ]:
df2 = df.sample(n=40, random_state=7).reset_index(drop=True)
lbl = sorted(df2["label"].unique())

y_actual = []
y_predicted = []

print(len(df2), "rows")

for i, j in df2.iterrows():
    text = j["Text"]
    y_actual.append(j["label"])
    y_predicted.append(predictor(text, lbl))

    if (i + 1) % 10 == 0:
        print("Complete:", i + 1)

accuracy = accuracy_score(y_actual, y_predicted)
f1 = f1_score(y_actual, y_predicted, average="macro")

print("LLM (zero-shot) accuracy:", accuracy)
print("LLM (zero-shot) macro F1:", f1)
print("Labels Predicted (uniques) :", len(set(y_predicted)))
print("Predicted labels:", list(y_predicted))

print("\nFirst 10 (actual label, model predicted label):")
print(list(zip(y_actual[:10], y_predicted[:10])))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


40 rows


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Complete: 10


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Complete: 20


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Complete: 30


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Complete: 40
LLM (zero-shot) accuracy: 0.2
LLM (zero-shot) macro F1: 0.19238095238095237
Labels Predicted (uniques) : 15
Predicted labels: ['middle_finger', 'smiling_face', 'middle_finger', 'face_with_tears_of_joy', 'face_with_tears_of_joy', 'face_with_tears_of_joy', 'backhand_index_pointing_right', 'smiling_face', 'smiling_face', 'smiling_face', 'face_with_tears_of_joy', 'smiling_face', 'sparkles', 'skull', 'smiling_face', 'thumbs_up', 'smiling_face', 'smiling_face', 'rolling_on_the_floor_laughing', 'folded_hands', 'folded_hands', 'pile_of_poo', 'smiling_face_with_hearts', 'folded_hands', 'backhand_index_pointing_right', 'loudly_crying_face', 'rabbit', 'folded_hands', 'smiling_face', 'smiling_face', 'smiling_face', 'face_with_tears_of_joy', 'smiling_face', 'smiling_face_with_hearts', 'folded_hands', 'smiling_face', 'smiling_face_with_sunglasses', 'face_with_tears_of_joy', 'cooking', 'rabbit']

First 10 (actual label, model predicted label):
[('backhand_index_pointing_right', 'middle_f

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# 1. Build confusion matrix for LLM
emojis = sorted(list(set(y_actual) | set(y_predicted)))
confusion = confusion_matrix(y_actual, y_predicted, labels=emojis)

df3 = pd.DataFrame(confusion, index=emojis, columns=emojis)
df3.index.name = "Actual"
df3.columns.name = "Predicted"

print("Confusion matrix shape:", df3.shape)

Confusion matrix shape: (25, 25)


In [ ]:
confused_emojis = []

for i,actual in enumerate(emojis):
  for j,prediction in enumerate(emojis):
    if i==j:
      continue

    n = df3.iat[i,j]
    if n>0:
      confused_emojis.append({"true": actual, "predicted":prediction, "number":int(n)})

df4 = pd.DataFrame(confused_emojis)
df4 = df4.sort_values("number", ascending=False).reset_index(drop=True)


print("\nMost Confused 20 Emojis\n", df4.head(20))


Most Confused 20 Emojis
                              true                      predicted  number
0                       thumbs_up                   folded_hands       3
1              loudly_crying_face         face_with_tears_of_joy       2
2                   saluting_face                   smiling_face       2
3        smiling_face_with_hearts                   smiling_face       2
4                  hatching_chick                   smiling_face       2
5                         cooking                   smiling_face       1
6                           ghost       smiling_face_with_hearts       1
7   backhand_index_pointing_right                  middle_finger       1
8       face_with_steam_from_nose         face_with_tears_of_joy       1
9          face_with_tears_of_joy  backhand_index_pointing_right       1
10       grinning_face_with_sweat                   smiling_face       1
11                 hatching_chick                   folded_hands       1
12                       